# **02477 Bayesian Machine Learning | PART 4: Code on Paper**

**Purpose:** Minimal, exam-style implementations of every core algorithm. Each block is self-contained — the exact pattern you would write on the exam. No imports needed to understand; pseudocode comments explain every line.

> **How to use:** Cover the code, write it from memory, check. Repeat until you can write each block in under 3 minutes.


---
## Table of Contents

1. [C1 | Log joint — universal recipe](#c1)
2. [C2 | MAP via numerical optimisation](#c2)
3. [C3 | Laplace approximation (full)](#c3)
4. [C4 | Grid approximation & normalisation](#c4)
5. [C5 | Posterior predictive via Monte Carlo](#c5)
6. [C6 | Metropolis-Hastings sampler](#c6)
7. [C7 | HMC leapfrog integrator](#c7)
8. [C8 | ELBO — analytical (mean-field Gaussian)](#c8)
9. [C9 | ELBO — BBVI with reparametrisation trick](#c9)
10. [C10 | Bayesian linear regression — exact posterior](#c10)
11. [C11 | GP regression — posterior predictive](#c11)
12. [C12 | CAVI update loop](#c12)


---
<a id='c1'></a>
<div class="alert alert-block alert-info">

## C1 | Log joint — universal recipe

**Used in:** every inference method. Always the starting point.
</div>

In [1]:
import jax.numpy as jnp
from jax.scipy.stats import norm
from scipy.stats import binom as binom_dist

# ==========================================
# 1. INPUTS (Change these for your exam!)
# ==========================================

# Example data
X = jnp.array([
    [1.0, 0.5],
    [1.0, -1.0],
    [1.0, 2.0]
])

y = jnp.array([1.2, 0.3, 2.1])

# Parameter vector
w = jnp.array([0.2, -0.4])

# Prior precision
alpha = 1.0

# Noise variance (Gaussian regression)
sigma2 = 1.0


# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================

# Log pdf of Gaussian
log_npdf = lambda x, m, v: (
    -0.5 * (x - m)**2 / v
    -0.5 * jnp.log(2 * jnp.pi * v)
)

# Sigmoid function
sigmoid = lambda x: 1.0 / (1.0 + jnp.exp(-x))


# ==========================================
# 3. PRIOR CALCULATION
# ==========================================

# Prior:
# w ~ N(0, alpha^{-1} I)

prior_variance = 1.0 / alpha

log_prior = jnp.sum(
    log_npdf(w, 0.0, prior_variance)
)


# ==========================================
# 4. LIKELIHOOD CALCULATION
# ==========================================

# ------------------------------------------------
# (a) Gaussian Regression
# y ~ N(Xw, sigma^2)
# ------------------------------------------------

f = X @ w

log_likelihood = jnp.sum(
    log_npdf(y, f, sigma2)
)


# ------------------------------------------------
# (b) Logistic Regression
# Uncomment instead of Gaussian
# y ~ Bernoulli(sigmoid(Xw))
# ------------------------------------------------

# f = X @ w

# log_likelihood = jnp.sum(
#     y * jnp.log(sigmoid(f))
#     + (1 - y) * jnp.log(1 - sigmoid(f))
# )


# ------------------------------------------------
# (c) Poisson Regression
# Uncomment instead of Gaussian
# y ~ Poisson(exp(Xw))
# ------------------------------------------------

# f = X @ w

# log_likelihood = jnp.sum(
#     y * f - jnp.exp(f)
# )
# # log(y!) dropped since it is constant


# ==========================================
# 5. LOG JOINT
# ==========================================

# log p(y, w)
# = log p(y|w) + log p(w)

log_joint = log_likelihood + log_prior


# ==========================================
# 6. RESULTS
# ==========================================

print("Log Prior:")
print(log_prior)

print("\nLog Likelihood:")
print(log_likelihood)

print("\nLog Joint:")
print(log_joint)

Log Prior:
-1.9378769

Log Likelihood:
-7.1668153

Log Joint:
-9.104692


---
<a id='c2'></a>
<div class="alert alert-block alert-info">

## C2 | MAP via numerical optimisation

**Used in:** Laplace approximation, plug-in predictions, as starting point for MCMC.
</div>

In [2]:
import jax
import jax.numpy as jnp
from jax import value_and_grad
from scipy.optimize import minimize


# ==========================================
# 1. INPUTS (Change these for your exam!)
# ==========================================

# Initial parameter guess
w_init = jnp.array([0.0, 0.0])

# Example data
X = jnp.array([
    [1.0, 0.5],
    [1.0, -1.0],
    [1.0, 2.0]
])

y = jnp.array([1.2, 0.3, 2.1])

# Prior precision
alpha = 1.0

# Noise variance
sigma2 = 1.0


# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================

# Gaussian log pdf
log_npdf = lambda x, m, v: (
    -0.5 * (x - m)**2 / v
    -0.5 * jnp.log(2 * jnp.pi * v)
)

# Sigmoid function
sigmoid = lambda x: 1.0 / (1.0 + jnp.exp(-x))


# ==========================================
# 3. DEFINE LOG JOINT
# ==========================================

# ------------------------------------------------
# PRIOR
# w ~ N(0, alpha^{-1} I)
# ------------------------------------------------

def log_joint(w):

    prior_variance = 1.0 / alpha

    log_prior = jnp.sum(
        log_npdf(w, 0.0, prior_variance)
    )

    # ------------------------------------------------
    # LIKELIHOOD
    # Gaussian Regression:
    # y ~ N(Xw, sigma^2)
    # ------------------------------------------------

    f = X @ w

    log_likelihood = jnp.sum(
        log_npdf(y, f, sigma2)
    )

    # ------------------------------------------------
    # Logistic Regression (alternative)
    # Uncomment instead
    # ------------------------------------------------

    # f = X @ w

    # log_likelihood = jnp.sum(
    #     y * jnp.log(sigmoid(f))
    #     + (1 - y) * jnp.log(1 - sigmoid(f))
    # )

    # ------------------------------------------------
    # Poisson Regression (alternative)
    # Uncomment instead
    # ------------------------------------------------

    # f = X @ w

    # log_likelihood = jnp.sum(
    #     y * f - jnp.exp(f)
    # )

    # ------------------------------------------------
    # LOG JOINT
    # ------------------------------------------------

    return log_likelihood + log_prior


# ==========================================
# 4. MAP OPTIMIZATION
# ==========================================

# MAP:
# argmax_w log p(y, w)
#
# scipy minimizes,
# so we minimize:
#
# -log p(y, w)

neg_log_joint_and_grad = value_and_grad(
    lambda w: -log_joint(w)
)

result = minimize(
    neg_log_joint_and_grad,
    x0=w_init,
    jac=True,          # function returns BOTH value and gradient
    method='BFGS'
)


# ==========================================
# 5. RESULTS
# ==========================================

# MAP estimate
w_MAP = result.x

print("MAP estimate:")
print(w_MAP)


# ==========================================
# 6. VERIFY THE MAP
# ==========================================

# Gradient should be close to zero at optimum

grad_at_MAP = jax.grad(log_joint)(w_MAP)

print("\nGradient at MAP:")
print(grad_at_MAP)

print("\nGradient norm:")
print(jnp.linalg.norm(grad_at_MAP))


# ==========================================
# 7. CLOSED-FORM MAP (Bayesian Linear Regression)
# ==========================================

# Only valid for Gaussian likelihood + Gaussian prior

# Equivalent to Ridge Regression

# lambda = alpha / beta
# where:
# beta = 1 / sigma2

beta = 1.0 / sigma2

D = X.shape[1]

closed_form_MAP = (
    jnp.linalg.inv(
        X.T @ X + (alpha / beta) * jnp.eye(D)
    )
    @ X.T
    @ y
)

print("\nClosed-form MAP:")
print(closed_form_MAP)

MAP estimate:
[0.69230767 0.55384613]

Gradient at MAP:
[1.1920929e-07 0.0000000e+00]

Gradient norm:
1.1920929e-07

Closed-form MAP:
[0.6923077 0.5538461]


---
<a id='c3'></a>
<div class="alert alert-block alert-info">

## C3 | Laplace approximation (full)

**Used in:** logistic regression, GP classification, Poisson regression, Bayesian NNs (LLLA).

**Output:** $q(\mathbf{w}) = \mathcal{N}(\mathbf{w}|\mathbf{w}_{\text{MAP}}, \mathbf{S})$ where $\mathbf{S} = (-\mathbf{H})^{-1}$
</div>

In [3]:
import jax
import jax.numpy as jnp
from jax import hessian, grad, value_and_grad
from scipy.optimize import minimize
from scipy.stats import norm


# ==========================================
# 1. INPUTS (Change these for your exam!)
# ==========================================

# Initial parameter guess
w_init = jnp.array([0.0, 0.0])

# Example data
X = jnp.array([
    [1.0, 0.5],
    [1.0, -1.0],
    [1.0, 2.0]
])

y = jnp.array([1.2, 0.3, 2.1])

# Prior precision
alpha = 1.0

# Noise variance
sigma2 = 1.0


# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================

# Gaussian log pdf
log_npdf = lambda x, m, v: (
    -0.5 * (x - m)**2 / v
    -0.5 * jnp.log(2 * jnp.pi * v)
)

# Sigmoid
sigmoid = lambda x: 1.0 / (1.0 + jnp.exp(-x))


# ==========================================
# 3. DEFINE LOG JOINT
# ==========================================

def log_joint(w):

    # --------------------------------------
    # PRIOR
    # w ~ N(0, alpha^{-1} I)
    # --------------------------------------

    prior_variance = 1.0 / alpha

    log_prior = jnp.sum(
        log_npdf(w, 0.0, prior_variance)
    )

    # --------------------------------------
    # LIKELIHOOD
    # Gaussian Regression
    # y ~ N(Xw, sigma²)
    # --------------------------------------

    f = X @ w

    log_likelihood = jnp.sum(
        log_npdf(y, f, sigma2)
    )

    # --------------------------------------
    # Logistic Regression (alternative)
    # Uncomment instead
    # --------------------------------------

    # f = X @ w

    # log_likelihood = jnp.sum(
    #     y * jnp.log(sigmoid(f))
    #     + (1 - y) * jnp.log(1 - sigmoid(f))
    # )

    # --------------------------------------
    # LOG JOINT
    # --------------------------------------

    return log_likelihood + log_prior


# ==========================================
# 4. STEP 1 — FIND MAP
# ==========================================

# Laplace approximation starts at the MAP

result = minimize(
    value_and_grad(lambda w: -log_joint(w)),
    x0=w_init,
    jac=True,
    method='BFGS'
)

# MAP estimate
w_MAP = jnp.array(result.x)

print("MAP estimate:")
print(w_MAP)


# ==========================================
# 5. STEP 2 — COMPUTE HESSIAN
# ==========================================

# Hessian:
# H = d²/dw² log p(y,w)

H = hessian(log_joint)(w_MAP)

print("\nHessian:")
print(H)


# ==========================================
# 6. STEP 3 — COMPUTE POSTERIOR COVARIANCE
# ==========================================

# Laplace approximation:
#
# q(w) = N(w_MAP, S)
#
# where:
#
# S = (-H)^(-1)

S = jnp.linalg.inv(-H)

print("\nPosterior covariance matrix S:")
print(S)


# ==========================================
# 7. INTERPRETATION
# ==========================================

# q(w) approximates:
#
# p(w|y)
#
# with a Gaussian centered at the MAP.


# ==========================================
# 8. POSTERIOR PREDICTIVE (REGRESSION)
# ==========================================

# Example test point

x_star = jnp.array([1.0, 1.5])

# Feature vector
phi_star = x_star

# Predictive mean
mu_f = phi_star @ w_MAP

# Epistemic uncertainty
var_f = phi_star @ S @ phi_star

# Total predictive variance
# (includes observation noise)

var_y = var_f + sigma2

print("\nRegression predictive mean:")
print(mu_f)

print("\nRegression predictive variance:")
print(var_y)


# ==========================================
# 9. POSTERIOR PREDICTIVE (CLASSIFICATION)
# ==========================================

# Probit approximation

classification_prob = norm.cdf(
    mu_f / jnp.sqrt(1 + (jnp.pi / 8) * var_f)
)

print("\nClassification probability p(y=1):")
print(classification_prob)


# ==========================================
# 10. SANITY CHECKS
# ==========================================

# --------------------------------------
# Gradient should be close to zero
# --------------------------------------

grad_at_MAP = grad(log_joint)(w_MAP)

print("\nGradient at MAP:")
print(grad_at_MAP)

print("\nGradient norm:")
print(jnp.linalg.norm(grad_at_MAP))


# --------------------------------------
# Hessian eigenvalues
# Should all be negative
# --------------------------------------

eigenvalues = jnp.linalg.eigvals(H)

print("\nHessian eigenvalues:")
print(eigenvalues)


# --------------------------------------
# Covariance matrix should be
# positive definite
# --------------------------------------

cov_eigenvalues = jnp.linalg.eigvals(S)

print("\nCovariance eigenvalues:")
print(cov_eigenvalues)

MAP estimate:
[0.69230765 0.5538461 ]

Hessian:
[[-4.   -1.5 ]
 [-1.5  -6.25]]

Posterior covariance matrix S:
[[ 0.2747253  -0.06593407]
 [-0.06593407  0.17582418]]

Regression predictive mean:
1.5230768

Regression predictive variance:
1.4725275

Classification probability p(y=1):
0.9190655394549032

Gradient at MAP:
[1.1920929e-07 0.0000000e+00]

Gradient norm:
1.1920929e-07

Hessian eigenvalues:
[-3.25+0.j -7.  +0.j]

Covariance eigenvalues:
[0.30769235+0.j 0.14285715+0.j]


---
<a id='c4'></a>
<div class="alert alert-block alert-info">

## C4 | Grid approximation & normalisation

**Used in:** Week 2 (logistic regression), any low-dimensional posterior ($D \leq 2$).

**Output:** Normalised grid weights $\pi_{ij}$ summing to 1.
</div>

In [4]:
import jax.numpy as jnp


# ==========================================
# 1. INPUTS (Change these for your exam!)
# ==========================================

# ------------------------------------------------
# Example 1D parameter grid
# ------------------------------------------------

theta_grid = jnp.linspace(-5, 5, 200)

# ------------------------------------------------
# Example 2D parameter grids
# ------------------------------------------------

alpha_grid = jnp.linspace(-3, 3, 100)
beta_grid  = jnp.linspace(-3, 3, 100)


# ==========================================
# 2. DEFINE LOG JOINT / LOG POSTERIOR
# ==========================================

# ------------------------------------------------
# 1D Example
# ------------------------------------------------

def log_joint_1d(theta):

    return (
        -(theta - 2.0)**2
        - 0.5 * theta**2
    )


# ------------------------------------------------
# 2D Example
# ------------------------------------------------

def log_joint_2d(alpha, beta):

    return (
        -(1 - alpha)**2
        - 20 * (beta - alpha**2)**2
        - alpha**2
        - beta**2
    )


# ==========================================
# 3. GRID APPROXIMATION — 1D
# ==========================================

# Evaluate log posterior on every grid point

log_vals_1d = jnp.array([
    log_joint_1d(theta)
    for theta in theta_grid
])

# --------------------------------------
# Numerical stability trick
# subtract largest value
# --------------------------------------

log_vals_1d = log_vals_1d - jnp.max(log_vals_1d)

# Convert from log space

unnormalized_pi_1d = jnp.exp(log_vals_1d)

# Normalization constant

Z_1d = jnp.sum(unnormalized_pi_1d)

# Normalize probabilities

pi_1d = unnormalized_pi_1d / Z_1d


# ==========================================
# 4. VERIFY NORMALIZATION
# ==========================================

print("1D probabilities sum to:")
print(jnp.sum(pi_1d))


# ==========================================
# 5. POSTERIOR SUMMARIES — 1D
# ==========================================

# --------------------------------------
# Posterior mean
# E[theta]
# --------------------------------------

posterior_mean = jnp.sum(
    theta_grid * pi_1d
)

# --------------------------------------
# Posterior second moment
# E[theta²]
# --------------------------------------

posterior_second_moment = jnp.sum(
    theta_grid**2 * pi_1d
)

# --------------------------------------
# Posterior variance
# Var(theta)
# --------------------------------------

posterior_variance = (
    posterior_second_moment
    - posterior_mean**2
)

print("\nPosterior mean:")
print(posterior_mean)

print("\nPosterior variance:")
print(posterior_variance)


# ==========================================
# 6. POSTERIOR PROBABILITY
# ==========================================

# Example:
# P(theta < 0)

prob_theta_less_than_0 = jnp.sum(
    pi_1d[theta_grid < 0]
)

print("\nP(theta < 0):")
print(prob_theta_less_than_0)


# ==========================================
# 7. POSTERIOR PREDICTIVE
# ==========================================

# Example:
# p(y*|theta) = N(theta, 1)

y_star = 1.0

predictive_terms = (
    (1 / jnp.sqrt(2 * jnp.pi))
    * jnp.exp(-0.5 * (y_star - theta_grid)**2)
)

# Approximation:
#
# p(y*|y) ≈ Σ p(y*|theta_i) pi_i

posterior_predictive = jnp.sum(
    predictive_terms * pi_1d
)

print("\nPosterior predictive:")
print(posterior_predictive)


# ==========================================
# 8. GRID APPROXIMATION — 2D
# ==========================================

# Create meshgrid

A, B = jnp.meshgrid(
    alpha_grid,
    beta_grid,
    indexing='ij'
)

# Evaluate log posterior everywhere

log_vals_2d = jnp.vectorize(
    log_joint_2d
)(A, B)

# Numerical stability

log_vals_2d = log_vals_2d - jnp.max(log_vals_2d)

# Convert from log space

unnormalized_pi_2d = jnp.exp(log_vals_2d)

# Normalize

pi_2d = (
    unnormalized_pi_2d
    / jnp.sum(unnormalized_pi_2d)
)

print("\n2D probabilities sum to:")
print(jnp.sum(pi_2d))


# ==========================================
# 9. MARGINAL DISTRIBUTIONS
# ==========================================

# Marginal over alpha

pi_alpha = pi_2d.sum(axis=1)

# Marginal over beta

pi_beta = pi_2d.sum(axis=0)

print("\nAlpha marginal shape:")
print(pi_alpha.shape)

print("\nBeta marginal shape:")
print(pi_beta.shape)


# ==========================================
# 10. POSTERIOR MEANS FROM 2D GRID
# ==========================================

# E[alpha]

mean_alpha = jnp.sum(
    alpha_grid * pi_alpha
)

# E[beta]

mean_beta = jnp.sum(
    beta_grid * pi_beta
)

print("\nPosterior mean alpha:")
print(mean_alpha)

print("\nPosterior mean beta:")
print(mean_beta)

1D probabilities sum to:
1.0000002

Posterior mean:
1.3333336

Posterior variance:
0.33333266

P(theta < 0):
0.0104404725

Posterior predictive:
0.33139443

2D probabilities sum to:
1.0000008

Alpha marginal shape:
(100,)

Beta marginal shape:
(100,)

Posterior mean alpha:
0.33432445

Posterior mean beta:
0.25297317


---
<a id='c5'></a>
<div class="alert alert-block alert-info">

## C5 | Posterior predictive via Monte Carlo

**Used in:** any model where we have posterior samples (MCMC, Laplace, BBVI, deep ensembles).

**Key formula:** $p(y^*|\mathbf{y}, x^*) \approx \frac{1}{S}\sum_{s=1}^S p(y^*|\mathbf{w}^{(s)}, x^*)$
</div>

In [5]:
import jax.numpy as jnp
from jax import random
from jax.scipy.special import logsumexp


# ==========================================
# 1. INPUTS (Change these for your exam!)
# ==========================================

# MAP estimate from Laplace approximation

w_MAP = jnp.array([1.0, -0.5])

# Posterior covariance matrix

S = jnp.array([
    [0.5, 0.1],
    [0.1, 0.3]
])

# Test input

x_star = jnp.array([1.0, 2.0])

# Number of Monte Carlo samples

S_samples = 1000

# Regression noise variance

sigma2 = 1.0

# Random seed

seed = 0

# Prediction mode
# 'regression' or 'classification'

mode = 'regression'


# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================

# Sigmoid

sigmoid = lambda x: 1.0 / (1.0 + jnp.exp(-x))

# Gaussian log pdf

log_npdf = lambda x, m, v: (
    -0.5 * (x - m)**2 / v
    -0.5 * jnp.log(2 * jnp.pi * v)
)


# ==========================================
# 3. FEATURE MAP
# ==========================================

# Example:
# phi(x) = x

def phi(x):
    return x


# ==========================================
# 4. SAMPLE FROM POSTERIOR
# ==========================================

# Approximate posterior:
#
# q(w) = N(w_MAP, S)

key = random.PRNGKey(seed)

w_samples = random.multivariate_normal(
    key,
    w_MAP,
    S,
    shape=(S_samples,)
)

# Shape:
# (S_samples, D)

print("Weight samples shape:")
print(w_samples.shape)


# ==========================================
# 5. COMPUTE LATENT FUNCTION SAMPLES
# ==========================================

# Compute:
#
# f* = phi(x*)^T w

phi_star = phi(x_star)

f_samples = w_samples @ phi_star

# Shape:
# (S_samples,)

print("\nFunction samples shape:")
print(f_samples.shape)


# ==========================================
# 6. REGRESSION PREDICTIVE
# ==========================================

if mode == 'regression':

    # --------------------------------------
    # Posterior predictive mean
    # --------------------------------------

    mu_pred = jnp.mean(f_samples)

    # --------------------------------------
    # Predictive variance
    #
    # epistemic + aleatoric
    # --------------------------------------

    var_pred = (
        jnp.var(f_samples)
        + sigma2
    )

    print("\nRegression predictive mean:")
    print(mu_pred)

    print("\nRegression predictive variance:")
    print(var_pred)


# ==========================================
# 7. CLASSIFICATION PREDICTIVE
# ==========================================

elif mode == 'classification':

    # --------------------------------------
    # Convert latent values to probabilities
    # --------------------------------------

    p_samples = sigmoid(f_samples)

    # --------------------------------------
    # Average probabilities
    # --------------------------------------

    p_pred = jnp.mean(p_samples)

    print("\nP(y*=1 | y, x*):")
    print(p_pred)


# ==========================================
# 8. LOG PREDICTIVE DENSITY (LPD)
# ==========================================

# Used for model comparison

# Example observed target

y_star = 1.5

# Compute:
#
# p(y*|y,x*)
# ≈ (1/S) Σ N(y* | f_s, sigma²)

log_probs = log_npdf(
    y_star,
    f_samples,
    sigma2
)

# Numerical stability with logsumexp

LPD = (
    logsumexp(log_probs)
    - jnp.log(S_samples)
)

print("\nLog Predictive Density:")
print(LPD)


# ==========================================
# 9. ANCESTRAL SAMPLING (POISSON)
# ==========================================

# Example:
# Poisson regression

# --------------------------------------
# Poisson means
# --------------------------------------

mu_samples = jnp.exp(f_samples)

# --------------------------------------
# Predictive samples
# --------------------------------------

key, subkey = random.split(key)

y_samples = random.poisson(
    subkey,
    mu_samples
)

print("\nPoisson predictive samples:")
print(y_samples[:10])

# --------------------------------------
# Monte Carlo estimate of expectation
# --------------------------------------

expected_y = jnp.mean(y_samples)

print("\nE[y*]:")
print(expected_y)

Weight samples shape:
(1000, 2)

Function samples shape:
(1000,)

Regression predictive mean:
-0.04760389

Regression predictive variance:
3.1299167

Log Predictive Density:
-1.8468871

Poisson predictive samples:
[42  0  0  0  0  0  3 24  9  0]

E[y*]:
2.6620002


---
<a id='c6'></a>
<div class="alert alert-block alert-info">

## C6 | Metropolis-Hastings sampler

**Used in:** Week 8 — sampling from any posterior $p(\theta|y)$.

**Key:** Only needs `log_joint` (not the normalised posterior). Symmetric Gaussian proposal → proposal ratio = 1.
</div>

In [6]:
import jax.numpy as jnp
from jax import random


# ==========================================
# 1. INPUTS (Change these for your exam!)
# ==========================================

# Initial parameter value

theta_init = jnp.array([0.0, 1.5])

# Proposal standard deviation
# Controls step size

tau = 0.5

# Number of MCMC iterations

num_iter = 5000

# Random seed

seed = 0


# ==========================================
# 2. DEFINE TARGET DISTRIBUTION
# ==========================================

# We only need:
#
# log p(y, theta)
#
# It does NOT need to be normalized

def log_target(theta):

    theta1 = theta[0]
    theta2 = theta[1]

    return (
        -(1 - theta1)**2
        - 20 * (theta2 - theta1**2)**2
        - theta1**2
        - theta2**2
    )


# ==========================================
# 3. INITIALIZE MCMC
# ==========================================

key = random.PRNGKey(seed)

# Dimension of parameter space

D = len(theta_init)

# Current state of chain

theta = theta_init

# Store samples

samples = [theta]

# Current log probability

log_p = log_target(theta)

# Count accepted proposals

n_accept = 0


# ==========================================
# 4. METROPOLIS-HASTINGS LOOP
# ==========================================

for _ in range(num_iter):

    # Split random keys

    key, key_prop, key_accept = random.split(key, 3)

    # --------------------------------------
    # Step 1:
    # Propose new parameter
    #
    # theta* = theta + N(0, tau² I)
    # --------------------------------------

    theta_star = (
        theta
        + tau * random.normal(
            key_prop,
            shape=(D,)
        )
    )

    # --------------------------------------
    # Step 2:
    # Evaluate target distribution
    # --------------------------------------

    log_p_star = log_target(theta_star)

    # --------------------------------------
    # Step 3:
    # Compute acceptance ratio
    #
    # log(r)
    # = log p(theta*)
    # - log p(theta)
    #
    # Proposal cancels because
    # Gaussian proposal is symmetric
    # --------------------------------------

    log_r = log_p_star - log_p

    # --------------------------------------
    # Step 4:
    # Draw uniform random number
    # --------------------------------------

    log_u = jnp.log(
        random.uniform(key_accept)
    )

    # --------------------------------------
    # Step 5:
    # Accept or reject
    # --------------------------------------

    if log_u < log_r:

        # Accept proposal

        theta = theta_star
        log_p = log_p_star

        n_accept += 1

    # Otherwise:
    # reject and stay at current theta

    samples.append(theta)


# ==========================================
# 5. CONVERT TO ARRAY
# ==========================================

samples = jnp.stack(samples)

print("Samples shape:")
print(samples.shape)


# ==========================================
# 6. ACCEPTANCE RATE
# ==========================================

acceptance_rate = n_accept / num_iter

print("\nAcceptance rate:")
print(acceptance_rate)


# ==========================================
# 7. REMOVE WARMUP / BURN-IN
# ==========================================

warmup = 1000

posterior_samples = samples[warmup:]

print("\nPosterior samples shape:")
print(posterior_samples.shape)


# ==========================================
# 8. POSTERIOR SUMMARIES
# ==========================================

posterior_mean = jnp.mean(
    posterior_samples,
    axis=0
)

posterior_std = jnp.std(
    posterior_samples,
    axis=0
)

print("\nPosterior mean:")
print(posterior_mean)

print("\nPosterior std:")
print(posterior_std)


# ==========================================
# 9. TRACE INFORMATION
# ==========================================

# First parameter chain

theta1_chain = posterior_samples[:, 0]

# Second parameter chain

theta2_chain = posterior_samples[:, 1]

print("\nTheta1 first 10 samples:")
print(theta1_chain[:10])

print("\nTheta2 first 10 samples:")
print(theta2_chain[:10])


# ==========================================
# 10. RULES OF THUMB
# ==========================================

# tau too small:
# -> high acceptance
# -> slow mixing
# -> highly correlated samples

# tau too large:
# -> low acceptance
# -> chain gets stuck

# Good target acceptance rates:
# ~44% in 1D
# ~23% in high dimensions

# Always inspect:
# - trace plots
# - histograms
# - autocorrelation

Samples shape:
(5001, 2)

Acceptance rate:
0.2312

Posterior samples shape:
(4001, 2)

Posterior mean:
[0.3641247 0.268893 ]

Posterior std:
[0.37634224 0.3260616 ]

Theta1 first 10 samples:
[ 0.07947722 -0.04777049  0.7356009   0.7356009   0.7356009   0.7356009
  0.7356009   0.45433944  0.45433944  0.31074303]

Theta2 first 10 samples:
[-0.10914885  0.03531258  0.358934    0.358934    0.358934    0.358934
  0.358934    0.15092936  0.15092936  0.03544386]


---
<a id='c7'></a>
<div class="alert alert-block alert-info">

## C7 | HMC leapfrog integrator

**Used in:** Week 9 — Hamiltonian Monte Carlo.

**Key equations:**
$\nu_{j+1/2} = \nu_j - \frac{\eta}{2}\nabla_\theta E(\theta_j)$, 
$\theta_{j+1} = \theta_j + \eta\nu_{j+1/2}$, 
$\nu_{j+1} = \nu_{j+1/2} - \frac{\eta}{2}\nabla_\theta E(\theta_{j+1})$
</div>

In [7]:
import jax.numpy as jnp
from jax import grad, random
from jax import jit


# ==========================================
# 1. INPUTS (Change these for your exam!)
# ==========================================

# Initial parameter value

theta_init = jnp.array([0.0, 1.5])

# HMC settings

step_size = 0.05       # eta
num_leapfrog = 20      # L
num_iter = 5000

# Random seed

seed = 0


# ==========================================
# 2. DEFINE TARGET DISTRIBUTION
# ==========================================

# Unnormalized log posterior

def log_target(theta):

    theta1 = theta[0]
    theta2 = theta[1]

    return (
        -(1 - theta1)**2
        - 20 * (theta2 - theta1**2)**2
        - theta1**2
        - theta2**2
    )


# ==========================================
# 3. DEFINE ENERGY FUNCTION
# ==========================================

# Potential energy:
#
# E(theta) = -log p(theta)

E = lambda theta: -log_target(theta)

# Gradient of potential energy

grad_E = jit(grad(E))


# ==========================================
# 4. LEAPFROG INTEGRATOR
# ==========================================

# Leapfrog approximates Hamiltonian dynamics

def leapfrog(theta, nu):

    for _ in range(num_leapfrog):

        # --------------------------------------
        # Half momentum step
        # --------------------------------------

        nu = (
            nu
            - 0.5 * step_size * grad_E(theta)
        )

        # --------------------------------------
        # Full position step
        # --------------------------------------

        theta = (
            theta
            + step_size * nu
        )

        # --------------------------------------
        # Half momentum step
        # --------------------------------------

        nu = (
            nu
            - 0.5 * step_size * grad_E(theta)
        )

    return theta, nu


# ==========================================
# 5. INITIALIZE HMC
# ==========================================

key = random.PRNGKey(seed)

theta = theta_init

# Dimension of parameter space

D = len(theta)

# Store samples

samples = [theta]

# Count accepted proposals

n_accept = 0


# ==========================================
# 6. HMC LOOP
# ==========================================

for _ in range(num_iter):

    # Split random keys

    key, key_nu, key_accept = random.split(key, 3)

    # --------------------------------------
    # Step 1:
    # Sample momentum
    #
    # nu ~ N(0, I)
    # --------------------------------------

    nu = random.normal(
        key_nu,
        shape=(D,)
    )

    # --------------------------------------
    # Step 2:
    # Current Hamiltonian
    #
    # H = E + K
    # --------------------------------------

    H_current = (
        E(theta)
        + 0.5 * jnp.sum(nu**2)
    )

    # --------------------------------------
    # Step 3:
    # Simulate Hamiltonian dynamics
    # --------------------------------------

    theta_prop, nu_prop = leapfrog(
        theta,
        nu
    )

    # --------------------------------------
    # Step 4:
    # Proposed Hamiltonian
    # --------------------------------------

    H_proposed = (
        E(theta_prop)
        + 0.5 * jnp.sum(nu_prop**2)
    )

    # --------------------------------------
    # Step 5:
    # Metropolis acceptance step
    # --------------------------------------

    log_accept = (
        -H_proposed
        + H_current
    )

    log_u = jnp.log(
        random.uniform(key_accept)
    )

    # --------------------------------------
    # Step 6:
    # Accept or reject
    # --------------------------------------

    if log_u < log_accept:

        theta = theta_prop

        n_accept += 1

    samples.append(theta)


# ==========================================
# 7. CONVERT TO ARRAY
# ==========================================

samples = jnp.stack(samples)

print("Samples shape:")
print(samples.shape)


# ==========================================
# 8. ACCEPTANCE RATE
# ==========================================

acceptance_rate = n_accept / num_iter

print("\nAcceptance rate:")
print(acceptance_rate)


# ==========================================
# 9. REMOVE WARMUP / BURN-IN
# ==========================================

warmup = 1000

posterior_samples = samples[warmup:]

print("\nPosterior samples shape:")
print(posterior_samples.shape)


# ==========================================
# 10. POSTERIOR SUMMARIES
# ==========================================

posterior_mean = jnp.mean(
    posterior_samples,
    axis=0
)

posterior_std = jnp.std(
    posterior_samples,
    axis=0
)

print("\nPosterior mean:")
print(posterior_mean)

print("\nPosterior std:")
print(posterior_std)


# ==========================================
# 11. TRACE INFORMATION
# ==========================================

theta1_chain = posterior_samples[:, 0]
theta2_chain = posterior_samples[:, 1]

print("\nTheta1 first 10 samples:")
print(theta1_chain[:10])

print("\nTheta2 first 10 samples:")
print(theta2_chain[:10])


# ==========================================
# 12. IMPORTANT HMC PROPERTIES
# ==========================================

# Leapfrog properties:
#
# 1. Reversible
# 2. Volume preserving
# 3. Approximately conserves Hamiltonian
# 4. Gives high acceptance rates

# If step_size too large:
# -> Hamiltonian changes too much
# -> low acceptance

# If step_size too small:
# -> accurate simulation
# -> expensive computation

# HMC usually mixes MUCH faster
# than random-walk Metropolis-Hastings

Samples shape:
(5001, 2)

Acceptance rate:
0.9848

Posterior samples shape:
(4001, 2)

Posterior mean:
[0.32974422 0.25071982]

Posterior std:
[0.39539614 0.32614577]

Theta1 first 10 samples:
[ 0.6143952   0.35526407  0.98749214 -0.29363552  0.6534381   0.44009176
  0.42032695 -0.08694296  0.86799914 -0.10863769]

Theta2 first 10 samples:
[ 0.5804983  -0.0750789   0.93898517  0.13796835  0.68046373  0.09250109
  0.35406634  0.04566972  0.4903656   0.08123446]


---
<a id='c8'></a>
<div class="alert alert-block alert-info">

## C8 | ELBO — analytical (mean-field Gaussian)

**Used in:** Week 10-11 — variational inference for Bayesian linear regression.

**Formula:** $\mathcal{L} = \mathbb{E}_q[\ln p(\mathbf{y}|\mathbf{w})] + \mathbb{E}_q[\ln p(\mathbf{w})] + \mathcal{H}[q]$
</div>

In [8]:
import jax.numpy as jnp
from jax import value_and_grad
from scipy.optimize import minimize


# ==========================================
# 1. INPUTS (Change these for your exam!)
# ==========================================

# Example dataset

X = jnp.array([
    [1.0, 0.5],
    [1.0, -1.0],
    [1.0, 2.0]
])

y = jnp.array([1.2, 0.3, 2.1])

# Number of parameters

D = X.shape[1]

# Noise variance

sigma2 = 1.0

# Prior variance

kappa2 = 1.0


# ==========================================
# 2. INITIAL VARIATIONAL PARAMETERS
# ==========================================

# Variational distribution:
#
# q(w) = Π_i N(w_i | m_i, v_i)

# Initial means

m = jnp.zeros(D)

# Initial log variances
#
# We optimize log(v)
# instead of v directly
# to guarantee positivity

log_v = jnp.zeros(D)


# ==========================================
# 3. HELPER FUNCTION
# ==========================================

# Gaussian log pdf

log_npdf = lambda x, m, v: (
    -0.5 * (x - m)**2 / v
    -0.5 * jnp.log(2 * jnp.pi * v)
)


# ==========================================
# 4. CONVERT LOG VARIANCES
# ==========================================

# Variances must be positive

v = jnp.exp(log_v)

print("Variances:")
print(v)


# ==========================================
# 5. EXPECTED LOG LIKELIHOOD
# ==========================================

# E_q[log p(y|w)]

# Residuals using variational means

residuals = y - X @ m

# Standard Gaussian likelihood term

expected_ll_main = jnp.sum(
    log_npdf(
        y,
        X @ m,
        sigma2
    )
)

# Extra variance correction term
#
# Comes from:
#
# E[w_i²] = m_i² + v_i

variance_correction = (
    -1 / (2 * sigma2)
    * jnp.sum(
        v * jnp.sum(X**2, axis=0)
    )
)

# Total expected log likelihood

expected_ll = (
    expected_ll_main
    + variance_correction
)

print("\nExpected log likelihood:")
print(expected_ll)


# ==========================================
# 6. EXPECTED LOG PRIOR
# ==========================================

# E_q[log p(w)]

expected_lp_main = jnp.sum(
    log_npdf(
        m,
        0.0,
        kappa2
    )
)

prior_variance_correction = (
    -1 / (2 * kappa2)
    * jnp.sum(v)
)

expected_lp = (
    expected_lp_main
    + prior_variance_correction
)

print("\nExpected log prior:")
print(expected_lp)


# ==========================================
# 7. ENTROPY OF q(w)
# ==========================================

# Mean-field Gaussian entropy:
#
# H[q]
# = 1/2 Σ log(2πev_i)

entropy = 0.5 * jnp.sum(
    jnp.log(
        2 * jnp.pi * jnp.e * v
    )
)

print("\nEntropy:")
print(entropy)


# ==========================================
# 8. COMPUTE ELBO
# ==========================================

# ELBO:
#
# ELBO
# =
# E_q[log p(y|w)]
# +
# E_q[log p(w)]
# +
# H[q]

ELBO = (
    expected_ll
    + expected_lp
    + entropy
)

print("\nELBO:")
print(ELBO)


# ==========================================
# 9. OPTIMIZE ELBO
# ==========================================

# Combine parameters:
#
# lambda = [m, log_v]

lam_init = jnp.concatenate([
    jnp.zeros(D),
    jnp.zeros(D)
])


# ==========================================
# 10. NEGATIVE ELBO FUNCTION
# ==========================================

# scipy minimizes,
# so we minimize:
#
# -ELBO

def neg_elbo(lam):

    # Split parameter vector

    m = lam[:D]
    log_v = lam[D:]

    # Convert variances

    v = jnp.exp(log_v)

    # --------------------------------------
    # Expected log likelihood
    # --------------------------------------

    expected_ll = (
        jnp.sum(
            log_npdf(
                y,
                X @ m,
                sigma2
            )
        )
        - 1/(2*sigma2)
        * jnp.sum(
            v * jnp.sum(X**2, axis=0)
        )
    )

    # --------------------------------------
    # Expected log prior
    # --------------------------------------

    expected_lp = (
        jnp.sum(
            log_npdf(
                m,
                0.0,
                kappa2
            )
        )
        - 1/(2*kappa2)
        * jnp.sum(v)
    )

    # --------------------------------------
    # Entropy
    # --------------------------------------

    entropy = (
        0.5
        * jnp.sum(
            jnp.log(
                2 * jnp.pi * jnp.e * v
            )
        )
    )

    # --------------------------------------
    # ELBO
    # --------------------------------------

    ELBO = (
        expected_ll
        + expected_lp
        + entropy
    )

    return -ELBO


# ==========================================
# 11. OPTIMIZATION
# ==========================================

result = minimize(
    value_and_grad(neg_elbo),
    lam_init,
    jac=True,
    method='BFGS'
)


# ==========================================
# 12. OPTIMAL VARIATIONAL PARAMETERS
# ==========================================

lam_opt = result.x

m_opt = lam_opt[:D]

v_opt = jnp.exp(
    lam_opt[D:]
)

print("\nOptimal variational means:")
print(m_opt)

print("\nOptimal variational variances:")
print(v_opt)

Variances:
[1. 1.]

Expected log likelihood:
-9.851815

Expected log prior:
-2.837877

Entropy:
2.837877

ELBO:
-9.851815

Optimal variational means:
[0.69230985 0.55385434]

Optimal variational variances:
[0.24999958 0.16000363]


---
<a id='c9'></a>
<div class="alert alert-block alert-info">

## C9 | ELBO — BBVI with reparametrisation trick

**Used in:** Week 11 — black-box VI for any model. Just swap in your `log_prior` and `log_lik`.

**Key:** reparametrisation $\mathbf{w} = \mathbf{m} + \mathbf{s} \circ \boldsymbol{\epsilon}$, $\boldsymbol{\epsilon} \sim \mathcal{N}(0, I)$ makes gradient flow through parameters.
</div>

In [9]:
import jax.numpy as jnp
from jax import random, value_and_grad


# ==========================================
# 1. INPUTS (Change these for your exam!)
# ==========================================

# Example dataset

X = jnp.array([
    [1.0, 0.5],
    [1.0, -1.0],
    [1.0, 2.0]
])

y = jnp.array([1.2, 0.3, 2.1])

# Number of parameters

D = X.shape[1]

# Monte Carlo samples

S_samples = 20

# Learning rate

lr = 0.01

# Number of optimization steps

num_steps = 500

# Random seed

seed = 0

# Noise variance

sigma2 = 1.0

# Prior variance

kappa2 = 1.0


# ==========================================
# 2. HELPER FUNCTION
# ==========================================

# Gaussian log pdf

log_npdf = lambda x, m, v: (
    -0.5 * (x - m)**2 / v
    -0.5 * jnp.log(2 * jnp.pi * v)
)


# ==========================================
# 3. DEFINE LOG LIKELIHOOD
# ==========================================

# Gaussian regression:
#
# y ~ N(Xw, sigma²)

def log_lik_fn(X, y, w):

    predictions = X @ w

    return jnp.sum(
        log_npdf(
            y,
            predictions,
            sigma2
        )
    )


# ==========================================
# 4. DEFINE LOG PRIOR
# ==========================================

# Prior:
#
# w ~ N(0, kappa² I)

def log_prior_fn(w):

    return jnp.sum(
        log_npdf(
            w,
            0.0,
            kappa2
        )
    )


# ==========================================
# 5. INITIAL VARIATIONAL PARAMETERS
# ==========================================

# Variational distribution:
#
# q(w) = N(m, diag(s²))

# We optimize:
#
# lambda = [m | log_s]

lam = jnp.zeros(2 * D)

print("Initial variational parameters:")
print(lam)


# ==========================================
# 6. SPLIT VARIATIONAL PARAMETERS
# ==========================================

m = lam[:D]

# Standard deviations must be positive

s = jnp.exp(lam[D:])

print("\nInitial means:")
print(m)

print("\nInitial std devs:")
print(s)


# ==========================================
# 7. REPARAMETERIZATION TRICK
# ==========================================

# Instead of:
#
# w ~ N(m, s²)
#
# write:
#
# w = m + s * eps
#
# eps ~ N(0, I)

key = random.PRNGKey(seed)

eps = random.normal(
    key,
    shape=(S_samples, D)
)

# Monte Carlo samples from q(w)

w_samples = m + s * eps

print("\nWeight samples shape:")
print(w_samples.shape)


# ==========================================
# 8. MONTE CARLO ESTIMATE
# ==========================================

# Estimate:
#
# E_q[log p(y,w)]

log_joint_samples = []

for i in range(S_samples):

    w = w_samples[i]

    log_joint = (
        log_lik_fn(X, y, w)
        + log_prior_fn(w)
    )

    log_joint_samples.append(log_joint)

log_joint_samples = jnp.array(log_joint_samples)

expected_log_joint = jnp.mean(log_joint_samples)

print("\nExpected log joint:")
print(expected_log_joint)


# ==========================================
# 9. ENTROPY TERM
# ==========================================

# Entropy of mean-field Gaussian:
#
# H[q]
# = 1/2 Σ log(2πes²)

entropy = 0.5 * jnp.sum(
    jnp.log(
        2 * jnp.pi * jnp.e * s**2
    )
)

print("\nEntropy:")
print(entropy)


# ==========================================
# 10. COMPUTE ELBO
# ==========================================

ELBO = (
    expected_log_joint
    + entropy
)

print("\nELBO:")
print(ELBO)


# ==========================================
# 11. FULL BBVI OBJECTIVE
# ==========================================

def neg_elbo(lam):

    # --------------------------------------
    # Split parameters
    # --------------------------------------

    m = lam[:D]

    s = jnp.exp(lam[D:])

    # --------------------------------------
    # Random samples
    # --------------------------------------

    key = random.PRNGKey(seed)

    eps = random.normal(
        key,
        shape=(S_samples, D)
    )

    w_samples = m + s * eps

    # --------------------------------------
    # Monte Carlo estimate
    # --------------------------------------

    log_joint_vals = []

    for i in range(S_samples):

        w = w_samples[i]

        log_joint = (
            log_lik_fn(X, y, w)
            + log_prior_fn(w)
        )

        log_joint_vals.append(log_joint)

    expected_log_joint = jnp.mean(
        jnp.array(log_joint_vals)
    )

    # --------------------------------------
    # Entropy
    # --------------------------------------

    entropy = 0.5 * jnp.sum(
        jnp.log(
            2 * jnp.pi * jnp.e * s**2
        )
    )

    ELBO = (
        expected_log_joint
        + entropy
    )

    # scipy / gradient descent minimizes
    # so return negative ELBO

    return -ELBO


# ==========================================
# 12. OPTIMIZATION LOOP
# ==========================================

neg_elbo_and_grad = value_and_grad(neg_elbo)

for step in range(num_steps):

    loss, g = neg_elbo_and_grad(lam)

    # Gradient descent on negative ELBO

    lam = lam - lr * g

    if step % 100 == 0:

        print(f"\nStep {step}")
        print(f"Negative ELBO: {loss}")


# ==========================================
# 13. FINAL VARIATIONAL DISTRIBUTION
# ==========================================

m_opt = lam[:D]

s_opt = jnp.exp(
    lam[D:]
)

print("\nOptimized variational means:")
print(m_opt)

print("\nOptimized variational std devs:")
print(s_opt)

Initial variational parameters:
[0. 0. 0. 0.]

Initial means:
[0. 0.]

Initial std devs:
[1. 1.]

Weight samples shape:
(20, 2)

Expected log joint:
-11.774566

Entropy:
2.837877

ELBO:
-8.936688

Step 0
Negative ELBO: 8.936688423156738

Step 100
Negative ELBO: 4.830098628997803

Step 200
Negative ELBO: 4.822037696838379

Step 300
Negative ELBO: 4.821908950805664

Step 400
Negative ELBO: 4.821907043457031

Optimized variational means:
[0.58317655 0.5303631 ]

Optimized variational std devs:
[0.5239938 0.3902051]


---
<a id='c10'></a>
<div class="alert alert-block alert-info">

## C10 | Bayesian linear regression — exact posterior

**Used in:** Weeks 3-4. Conjugate model — closed-form posterior, no optimisation needed.
</div>

In [10]:
import jax.numpy as jnp


# ==========================================
# 1. INPUTS (Change these for your exam!)
# ==========================================

# Design matrix
#
# Shape:
# (N, D)

Phi = jnp.array([
    [1.0, 0.5],
    [1.0, -1.0],
    [1.0, 2.0]
])

# Targets
#
# Shape:
# (N,)

y = jnp.array([1.2, 0.3, 2.1])

# Prior precision
#
# alpha = 1 / prior_variance

alpha = 1.0

# Noise precision
#
# beta = 1 / sigma²

beta = 1.0


# ==========================================
# 2. BASIC DIMENSIONS
# ==========================================

N, D = Phi.shape

print("Number of datapoints N:")
print(N)

print("\nNumber of parameters D:")
print(D)


# ==========================================
# 3. PRIOR
# ==========================================

# Prior:
#
# w ~ N(0, alpha^{-1} I)

prior_covariance = (
    1 / alpha
) * jnp.eye(D)

print("\nPrior covariance:")
print(prior_covariance)


# ==========================================
# 4. POSTERIOR COVARIANCE
# ==========================================

# Exact posterior covariance:
#
# S =
# (alpha I + beta Phi^T Phi)^(-1)

S = jnp.linalg.inv(
    alpha * jnp.eye(D)
    + beta * Phi.T @ Phi
)

print("\nPosterior covariance S:")
print(S)


# ==========================================
# 5. POSTERIOR MEAN
# ==========================================

# Exact posterior mean:
#
# m = beta S Phi^T y

m = (
    beta
    * S
    @ Phi.T
    @ y
)

print("\nPosterior mean m:")
print(m)


# ==========================================
# 6. INTERPRETATION
# ==========================================

# Posterior:
#
# p(w|y)
# =
# N(m, S)

# For Bayesian linear regression,
# the posterior is EXACTLY Gaussian.


# ==========================================
# 7. TEST INPUTS
# ==========================================

# New feature vectors
#
# Shape:
# (P, D)

Phi_star = jnp.array([
    [1.0, 0.0],
    [1.0, 1.5],
    [1.0, -2.0]
])


# ==========================================
# 8. PREDICT LATENT FUNCTION
# ==========================================

# Latent function:
#
# f* = Phi* w

# Posterior mean:
#
# E[f*] = Phi* m

mu_f = (
    Phi_star @ m
).ravel()

# Posterior variance:
#
# Var(f*)
# =
# Phi* S Phi*^T

var_f = jnp.diag(
    Phi_star
    @ S
    @ Phi_star.T
)

print("\nLatent predictive mean:")
print(mu_f)

print("\nLatent predictive variance:")
print(var_f)


# ==========================================
# 9. PREDICT OBSERVATIONS
# ==========================================

# Observation model:
#
# y* = f* + noise

# Add aleatoric uncertainty:
#
# Var(y*)
# =
# Var(f*)
# + 1/beta

var_y = (
    var_f
    + 1 / beta
)

print("\nPredictive variance for y*:")
print(var_y)


# ==========================================
# 10. FULL PREDICTIVE DISTRIBUTION
# ==========================================

# Predictive distribution:
#
# p(y*|y)
# =
# N(mu_f, var_y)

print("\nPredictive distribution:")
print("Mean:")
print(mu_f)

print("\nVariance:")
print(var_y)


# ==========================================
# 11. MARGINAL LIKELIHOOD
# ==========================================

# Covariance of y:
#
# C =
# beta^{-1} I
# +
# alpha^{-1} Phi Phi^T

C = (
    (1 / beta) * jnp.eye(N)
    + (1 / alpha) * Phi @ Phi.T
)

print("\nMarginal covariance C:")
print(C)


# ==========================================
# 12. LOG MARGINAL LIKELIHOOD
# ==========================================

# log p(y)
# =
# -N/2 log(2π)
# -1/2 log|C|
# -1/2 y^T C^{-1} y

sign, logdet = jnp.linalg.slogdet(C)

log_marginal_likelihood = (
    -N/2 * jnp.log(2 * jnp.pi)
    -0.5 * logdet
    -0.5 * y.T @ jnp.linalg.inv(C) @ y
)

print("\nLog marginal likelihood:")
print(log_marginal_likelihood)


# ==========================================
# 13. IMPORTANT LIMITS
# ==========================================

# alpha -> 0
#
# Flat prior
#
# Posterior mean approaches:
#
# (Phi^T Phi)^(-1) Phi^T y
#
# = MLE / least squares solution

# beta -> 0
#
# Infinite noise
#
# Posterior mean approaches:
#
# m -> 0
#
# because data becomes uninformative

# In Bayesian linear regression:
#
# MAP = posterior mean
#
# because posterior is Gaussian


# ==========================================
# 14. UNCERTAINTY INTERPRETATION
# ==========================================

# Epistemic uncertainty:
#
# Var(f*)
# =
# Phi* S Phi*^T
#
# decreases as N increases

# Aleatoric uncertainty:
#
# 1 / beta
#
# observation noise floor

# Total predictive uncertainty:
#
# Var(y*)
# =
# Var(f*)
# + 1/beta

Number of datapoints N:
3

Number of parameters D:
2

Prior covariance:
[[1. 0.]
 [0. 1.]]

Posterior covariance S:
[[ 0.2747253  -0.06593407]
 [-0.06593407  0.17582418]]

Posterior mean m:
[0.6923077 0.5538461]

Latent predictive mean:
[ 0.6923077   1.5230769  -0.41538453]

Latent predictive variance:
[0.2747253  0.47252747 1.2417583 ]

Predictive variance for y*:
[1.2747253 1.4725275 2.2417583]

Predictive distribution:
Mean:
[ 0.6923077   1.5230769  -0.41538453]

Variance:
[1.2747253 1.4725275 2.2417583]

Marginal covariance C:
[[ 2.25  0.5   2.  ]
 [ 0.5   3.   -1.  ]
 [ 2.   -1.    6.  ]]

Log marginal likelihood:
-4.79679


---
<a id='c11'></a>
<div class="alert alert-block alert-info">

## C11 | GP (Gaussian Process) regression — posterior predictive

**Used in:** Weeks 5-6. Non-parametric model — posterior over functions.
</div>

In [11]:
import jax.numpy as jnp


# ==========================================
# 1. INPUTS (Change these for your exam!)
# ==========================================

# Training inputs
#
# Shape:
# (N, D)

X_train = jnp.array([
    [0.0],
    [1.0],
    [2.0],
    [3.0]
])

# Training targets
#
# Shape:
# (N,)

y_train = jnp.array([
    0.2,
    0.9,
    2.1,
    2.9
])

# Test inputs
#
# Shape:
# (P, D)

X_star = jnp.array([
    [0.5],
    [1.5],
    [2.5]
])

# Kernel hyperparameters

kappa = 1.0      # signal std
ell = 1.0        # lengthscale

# Noise variance

sigma2 = 0.1

# Small numerical stability term

jitter = 1e-8


# ==========================================
# 2. SQUARED EXPONENTIAL KERNEL
# ==========================================

# Kernel:
#
# k(x,x')
# =
# kappa² exp(-||x-x'||² / 2ell²)

# Compute pairwise squared distances

dists = jnp.sum(
    (
        X_train[:, None, :]
        - X_train[None, :, :]
    )**2,
    axis=-1
)

# Training covariance matrix

K = (
    kappa**2
    * jnp.exp(
        -dists / (2 * ell**2)
    )
)

# Add jitter to diagonal

K = K + jitter * jnp.eye(len(X_train))

print("Kernel matrix K:")
print(K)


# ==========================================
# 3. TRAINING COVARIANCE
# ==========================================

# Noisy covariance:
#
# C = K + sigma² I

C = (
    K
    + sigma2 * jnp.eye(len(X_train))
)

print("\nTraining covariance C:")
print(C)


# ==========================================
# 4. CROSS-COVARIANCE
# ==========================================

# k_star:
#
# covariance between:
# test points and training points

dists_star = jnp.sum(
    (
        X_star[:, None, :]
        - X_train[None, :, :]
    )**2,
    axis=-1
)

k_star = (
    kappa**2
    * jnp.exp(
        -dists_star / (2 * ell**2)
    )
)

print("\nCross covariance k_star:")
print(k_star)


# ==========================================
# 5. TEST COVARIANCE
# ==========================================

# Prior covariance at test points

dists_test = jnp.sum(
    (
        X_star[:, None, :]
        - X_star[None, :, :]
    )**2,
    axis=-1
)

K_star = (
    kappa**2
    * jnp.exp(
        -dists_test / (2 * ell**2)
    )
)

K_star = K_star + jitter * jnp.eye(len(X_star))

print("\nTest covariance K_star:")
print(K_star)


# ==========================================
# 6. POSTERIOR MEAN
# ==========================================

# Posterior mean:
#
# mu_f
# =
# k_star C^{-1} y

# Solve:
#
# C alpha = y

alpha = jnp.linalg.solve(
    C,
    y_train
)

# Posterior mean

mu_f = (
    k_star @ alpha
).ravel()

print("\nPosterior mean mu_f:")
print(mu_f)


# ==========================================
# 7. POSTERIOR COVARIANCE
# ==========================================

# Posterior covariance:
#
# Sigma_f
# =
# K_star
# -
# k_star C^{-1} k_star^T

v = jnp.linalg.solve(
    C,
    k_star.T
)

Sigma_f = (
    K_star
    - k_star @ v
)

print("\nPosterior covariance Sigma_f:")
print(Sigma_f)


# ==========================================
# 8. PREDICTIVE VARIANCE
# ==========================================

# Latent function variance
#
# epistemic uncertainty only

var_f = jnp.diag(Sigma_f)

print("\nLatent predictive variance:")
print(var_f)


# ==========================================
# 9. OBSERVED PREDICTIVE VARIANCE
# ==========================================

# Add observation noise:
#
# Var(y*)
# =
# Var(f*)
# + sigma²

var_y = (
    var_f
    + sigma2
)

print("\nObserved predictive variance:")
print(var_y)


# ==========================================
# 10. PREDICTIVE DISTRIBUTION
# ==========================================

# Latent:
#
# f*|y
# ~
# N(mu_f, Sigma_f)

# Observed:
#
# y*|y
# ~
# N(mu_f, var_y)

print("\nPredictive mean:")
print(mu_f)

print("\nPredictive variance:")
print(var_y)


# ==========================================
# 11. LOG MARGINAL LIKELIHOOD
# ==========================================

# log p(y|theta)
#
# =
# -N/2 log(2π)
# -1/2 log|C|
# -1/2 y^T C^{-1} y

N = len(X_train)

sign, logdet = jnp.linalg.slogdet(C)

log_marginal_likelihood = (
    -N/2 * jnp.log(2 * jnp.pi)
    -0.5 * logdet
    -0.5 * y_train.T @ jnp.linalg.solve(C, y_train)
)

print("\nLog marginal likelihood:")
print(log_marginal_likelihood)


# ==========================================
# 12. IMPORTANT GP PROPERTIES
# ==========================================

# Near training data:
# -> posterior variance becomes small

# Far from training data:
# -> posterior variance approaches:
#
# k(x*,x*)
#
# the prior variance

# Latent predictions:
#
# f*
#
# contain only epistemic uncertainty

# Observed predictions:
#
# y*
#
# contain:
# - epistemic uncertainty
# - aleatoric uncertainty sigma²

Kernel matrix K:
[[1.         0.60653067 0.13533528 0.011109  ]
 [0.60653067 1.         0.60653067 0.13533528]
 [0.13533528 0.60653067 1.         0.60653067]
 [0.011109   0.13533528 0.60653067 1.        ]]

Training covariance C:
[[1.1        0.60653067 0.13533528 0.011109  ]
 [0.60653067 1.1        0.60653067 0.13533528]
 [0.13533528 0.60653067 1.1        0.60653067]
 [0.011109   0.13533528 0.60653067 1.1       ]]

Cross covariance k_star:
[[0.8824969  0.8824969  0.32465246 0.04393693]
 [0.32465246 0.8824969  0.8824969  0.32465246]
 [0.04393693 0.32465246 0.8824969  0.8824969 ]]

Test covariance K_star:
[[1.         0.60653067 0.13533528]
 [0.60653067 1.         0.60653067]
 [0.13533528 0.60653067 1.        ]]

Posterior mean mu_f:
[0.46810234 1.4170579  2.568682  ]

Posterior covariance Sigma_f:
[[ 0.08223009  0.0123229  -0.00537406]
 [ 0.01232284  0.07844543  0.0123229 ]
 [-0.00537407  0.0123229   0.08223009]]

Latent predictive variance:
[0.08223009 0.07844543 0.08223009]

Observed

---
<a id='c12'></a>
<div class="alert alert-block alert-info">

## C12 | CAVI update loop

**Used in:** Week 10 — coordinate ascent variational inference.

**Pattern:** Alternate between updating $q(\mu)$ and $q(\tau)$ until the ELBO converges.
</div>

In [12]:
import jax.numpy as jnp


# ==========================================
# 1. INPUTS (Change these for your exam!)
# ==========================================

# Observed data

x = jnp.array([
    1.2,
    0.7,
    1.5,
    0.9,
    1.1
])

# Prior hyperparameters

mu0 = 0.0
lambda0 = 1.0

a0 = 1.0
b0 = 1.0

# Number of CAVI iterations

num_iter = 100


# ==========================================
# 2. BASIC STATISTICS
# ==========================================

N = len(x)

xbar = jnp.mean(x)

x2bar = jnp.mean(x**2)

print("Number of datapoints:")
print(N)

print("\nSample mean:")
print(xbar)

print("\nSample second moment:")
print(x2bar)


# ==========================================
# 3. MODEL
# ==========================================

# Likelihood:
#
# x_n ~ N(mu, tau^{-1})

# Prior on mu:
#
# mu ~ N(mu0, (lambda0*tau)^(-1))

# Prior on precision:
#
# tau ~ Gamma(a0, b0)


# ==========================================
# 4. VARIATIONAL FAMILY
# ==========================================

# Mean-field approximation:
#
# q(mu, tau)
# =
# q(mu) q(tau)

# Variational distributions:
#
# q(mu)
# =
# N(m, v)
#
# q(tau)
# =
# Gamma(a, b)


# ==========================================
# 5. INITIALIZE VARIATIONAL PARAMETERS
# ==========================================

# q(mu)

m = 0.0
v = 1.0

# q(tau)

a = a0
b = b0

print("\nInitial m:")
print(m)

print("\nInitial v:")
print(v)

print("\nInitial a:")
print(a)

print("\nInitial b:")
print(b)


# ==========================================
# 6. CAVI LOOP
# ==========================================

for step in range(num_iter):

    # ======================================
    # UPDATE q*(mu)
    # ======================================

    # Expected precision:
    #
    # E[tau]
    # =
    # a / b

    E_tau = a / b

    # --------------------------------------
    # Variational variance
    # --------------------------------------

    v = 1.0 / (
        (N + lambda0)
        * E_tau
    )

    # --------------------------------------
    # Variational mean
    # --------------------------------------

    m = (
        N * xbar
        + lambda0 * mu0
    ) / (
        N + lambda0
    )

    # ======================================
    # UPDATE q*(tau)
    # ======================================

    # Expectations under q(mu)

    E_mu = m

    E_mu2 = (
        m**2
        + v
    )

    # --------------------------------------
    # Gamma shape parameter
    # --------------------------------------

    a = (
        a0
        + (N + 1) / 2.0
    )

    # --------------------------------------
    # Gamma rate parameter
    # --------------------------------------

    b = (
        b0
        + 0.5 * N * x2bar
        - E_mu * (
            N * xbar
            + lambda0 * mu0
        )
        + 0.5 * E_mu2 * (
            N + lambda0
        )
        + 0.5 * lambda0 * mu0**2
    )

    # --------------------------------------
    # Print progress occasionally
    # --------------------------------------

    if step % 20 == 0:

        print(f"\nStep {step}")

        print("m =", m)
        print("v =", v)
        print("a =", a)
        print("b =", b)


# ==========================================
# 7. FINAL VARIATIONAL DISTRIBUTIONS
# ==========================================

print("\nFinal variational parameters")

print("\nm:")
print(m)

print("\nv:")
print(v)

print("\na:")
print(a)

print("\nb:")
print(b)


# ==========================================
# 8. EXPECTATIONS
# ==========================================

# Expected precision

E_tau = a / b

# Expected mu

E_mu = m

# Expected mu²

E_mu2 = m**2 + v

print("\nE[tau]:")
print(E_tau)

print("\nE[mu]:")
print(E_mu)

print("\nE[mu²]:")
print(E_mu2)


# ==========================================
# 9. INTERPRET VARIATIONAL DISTRIBUTIONS
# ==========================================

# q(mu)
#
# =
# N(m, v)

print("\nq(mu) = N(m,v)")

print("Mean:")
print(m)

print("Variance:")
print(v)

# q(tau)
#
# =
# Gamma(a,b)

print("\nq(tau) = Gamma(a,b)")

print("Shape:")
print(a)

print("Rate:")
print(b)


# ==========================================
# 10. GENERAL CAVI RULE
# ==========================================

# General update:
#
# log q*(z_j)
# =
# E_q(-j)[log p(x,z)]
# + const
#
# where:
#
# q(-j)
#
# means expectation over all
# OTHER latent variables.

# Procedure:
#
# 1. Keep only terms involving z_j
# 2. Take expectations over others
# 3. Match functional form
# 4. Read off updated parameters

Number of datapoints:
5

Sample mean:
1.08

Sample second moment:
1.2400001

Initial m:
0.0

Initial v:
1.0

Initial a:
1.0

Initial b:
1.0

Step 0
m = 0.90000004
v = 0.16666666666666666
a = 4.0
b = 2.1700006

Step 20
m = 0.90000004
v = 0.079523824
a = 4.0
b = 1.9085717

Step 40
m = 0.90000004
v = 0.079523824
a = 4.0
b = 1.9085717

Step 60
m = 0.90000004
v = 0.079523824
a = 4.0
b = 1.9085717

Step 80
m = 0.90000004
v = 0.079523824
a = 4.0
b = 1.9085717

Final variational parameters

m:
0.90000004

v:
0.079523824

a:
4.0

b:
1.9085717

E[tau]:
2.095808

E[mu]:
0.90000004

E[mu²]:
0.88952386

q(mu) = N(m,v)
Mean:
0.90000004
Variance:
0.079523824

q(tau) = Gamma(a,b)
Shape:
4.0
Rate:
1.9085717


---

## **Quick Reference: What to write on the exam**

| Task | Key line(s) to write |
|---|---|
| Log joint | `log p(y,w) = sum_n log p(y_n|w) + log p(w)` |
| MAP | `w_MAP = argmax_w log_joint(w)` → gradient = 0 |
| Laplace | `S = inv(-H)`, `H = Hessian of log_joint at w_MAP` |
| Grid normalise | `pi = exp(log_joint - max) / sum(exp(log_joint - max))` |
| MH accept | `log_r = log_joint(theta*) - log_joint(theta)`, accept if `log(u) < log_r` |
| HMC half-step | `nu = nu - (eta/2) * grad_E(theta)` |
| HMC full-step | `theta = theta + eta * nu` |
| ELBO entropy | `H[q] = 0.5 * sum_i log(2 pi e v_i)` |
| BBVI reparam | `w = m + s * eps`, `eps ~ N(0,I)` |
| BLR posterior | `S = inv(alpha*I + beta*Phi.T@Phi)`, `m = beta*S@Phi.T@y` |
| GP posterior mean | `mu = k_star @ solve(K + sigma2*I, y)` |
| GP posterior var | `Sigma = K_star - k_star @ solve(C, k_star.T)` |
| CAVI update q(mu) | `v = 1/((N+lambda0)*E[tau])`, `m = (N*xbar + lambda0*mu0)/(N+lambda0)` |
| Probit approx | `p(y*=1) = Phi(mu_f / sqrt(8/pi + var_f))` |
